# B9 — Index-vector quantization test (`quant_int8`)

**Question:** the quantization that actually pays at scale - storing the
Qdrant vectors themselves as int8 (768 bytes instead of 3 KB per photo,
4x index memory) - what does it cost in recall, and does fp32
**rescoring of the top-20** recover it (Qdrant's standard pattern)?

**Method:** per-vector symmetric int8 scalar quantization of the gallery
(both the SigLIP-native and the adapter-produced vectors, since the
shared index holds both kinds); B3 retrieval protocol; then two-stage
search: int8 shortlist top-20 -> fp32 rescore.

**Pass criteria:** int8-only within 1 point of fp32 at every K; rescored
identical to fp32. Runtime: seconds; needs `pairs.npz` + `adapter.npz`.

In [ ]:
# Storage setup (same convention as all stages)
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['DATA_DIR'] = '/content/drive/MyDrive/convergence_experiment'
print('DATA_DIR =', os.environ['DATA_DIR'])

In [ ]:
import numpy as np
from pathlib import Path
DATA_DIR = Path(os.environ['DATA_DIR'])

def l2n(X):
    return X / (np.linalg.norm(X, axis=-1, keepdims=True) + 1e-9)

def recall(sim, ks=(1, 5, 10)):
    ranks = (-sim).argsort(axis=1)
    n = sim.shape[0]
    return {k: float((ranks[:, :k] == np.arange(n)[:, None]).any(1).mean())
            for k in ks}

def report(name, r, ceil=None):
    line = f"{name:<38} " + "  ".join(f"R@{k}={r[k]:.3f}" for k in (1, 5, 10))
    if ceil:
        pct = min(100 * r[k] / max(ceil[k], 1e-9) for k in (1, 5, 10))
        line += f"   (worst-K {pct:.1f}% of ref)"
    print(line)

pairs = np.load(DATA_DIR / 'pairs.npz')
ad = np.load(DATA_DIR / 'adapter.npz')
te = ad['eval_idx']
W = ad['W_ridge'].astype(np.float32)

txt = pairs['sig_txt'][te]
gallery_fp = {
    'server-indexed (SigLIP native)': pairs['sig_img'][te],
    'phone-indexed (adapter)': l2n(pairs['mob_img'][te] @ W),
}

def quant_vec_int8(V):
    """Per-vector symmetric int8: v ~ q * scale."""
    scale = np.abs(V).max(axis=1, keepdims=True) / 127.0
    q = np.round(V / scale).clip(-127, 127).astype(np.int8)
    return q, scale

for name, G in gallery_fp.items():
    q, s = quant_vec_int8(G)
    G8 = q.astype(np.float32) * s
    per_photo = q.shape[1] + 4                    # int8 payload + fp32 scale
    print(f'\n=== {name} ===  ({per_photo} bytes/photo vs '
          f'{G.shape[1]*4} fp32)')
    ceil = recall(txt @ G.T)
    r8 = recall(txt @ G8.T)
    report('fp32 gallery', ceil)
    report('int8 gallery', r8, ceil)

    # two-stage: int8 shortlist top-20, fp32 rescore
    sims8 = txt @ G8.T
    top20 = (-sims8).argsort(axis=1)[:, :20]
    n = len(te)
    resc = np.full_like(sims8, -1e9)
    rows = np.repeat(np.arange(n), 20)
    cols = top20.ravel()
    resc[rows, cols] = (txt[rows] * G[cols]).sum(1)
    rr = recall(resc)
    report('int8 shortlist + fp32 rescore', rr, ceil)
    ok8 = all(abs(r8[k]-ceil[k]) <= 0.01 for k in (1, 5, 10))
    okr = all(abs(rr[k]-ceil[k]) <= 0.001 for k in (1, 5, 10))
    print('int8-only:', 'PASS (<=1pt)' if ok8 else 'degraded - use rescoring')
    print('rescored :', 'PASS (matches fp32)' if okr else 'CHECK')

**Interpretation guide.** Scalar int8 on unit-norm 768-d vectors is
usually near-lossless for ranking, and top-20 fp32 rescoring closes any
residual gap at negligible cost (20 dot products per query). If both
galleries pass, the deployment recipe is: enable Qdrant scalar
quantization (int8) with rescoring on, cutting index memory ~4x for the
whole two-tier corpus - the quantization decision with actual leverage,
unlike the adapter's 0.4 MB. Testing both vector kinds matters because
the shared index stores server-native and adapter-produced vectors side
by side; both must survive the same compression.